<a href="https://colab.research.google.com/github/myriosMin/SP500-News-Sentiment-Analysis/blob/main/Stocks_Analysis%5BHabib%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
# Machine Learning
from sklearn.linear_model import LinearRegression

In [ ]:
#this cell may take a while
import requests
!pip install opendatasets
import opendatasets as od
!pip install yfinance
import yfinance as yf

# Data Loading
In this section, we will load the dataset into our environment.

In [ ]:
# Define the start and end dates
start_date = dt.datetime(2008, 1, 1)
end_date = dt.datetime.now()

# URL of the Wikipedia page for the list of S&P 500 companies
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

# Read the data from the Wikipedia page
tables = pd.read_html(url)

# The first table on the page contains the S&P 500 companies
com_list = tables[0]

# Display the first few rows of the DataFrame
com_list

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [ ]:
stocks_data = None #initialise the stocks dataframe

#downloading all the data for each company
for i in com_list['Symbol']:
  try:
    temp_df = yf.download(i, start=start_date, end=end_date)
    temp_df['Company'] = i
    if stocks_data is None:
        stocks_data = temp_df
    else:
        stocks_data = pd.concat([stocks_data, temp_df])
  except:
    pass

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%*******

$BF.B: possibly delisted; No price data found  (1d 2008-01-01 00:00:00 -> 2024-08-12 15:10:48.159209)



[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%******

# Data Exploration
Here, we perform an initial exploration of the dataset to understand its structure and identify any immediate issues.

In [ ]:
#Viewing the first 5 entries of stocks data
stocks_data.head(5)

,Open,High,Low,Close,Adj Close,Volume,Company
Date,,,,,,,
2008-01-02,70.434784,70.869568,68.754181,69.155518,41.474419,5326625.0,MMM
2008-01-03,69.247490,69.799332,68.871239,69.147156,41.469402,3258024.0,MMM
2008-01-04,68.561874,69.113708,68.185616,68.361206,40.998039,4329879.0,MMM
2008-01-07,68.645485,68.687294,67.533447,67.817726,40.672108,5051665.0,MMM
2008-01-08,68.018394,68.620399,66.973244,67.065216,40.220818,6830595.0,MMM


# Data Cleaning
This section covers the data cleaning process, including handling missing values, outliers, and duplicates.

In [ ]:
# dropping open high low close columns
stocks_data = stocks_data.drop(['Open', 'High', 'Low', 'Close'], axis=1)

In [ ]:
#Verifying the stocks data
stocks_data

,Adj Close,Volume,Company
Date,,,
2008-01-02,41.474419,5326625.0,MMM
2008-01-03,41.469402,3258024.0,MMM
2008-01-04,40.998039,4329879.0,MMM
2008-01-07,40.672108,5051665.0,MMM
2008-01-08,40.220818,6830595.0,MMM
...,...,...,...
2024-08-06,185.289993,3832500.0,ZTS
2024-08-07,184.770004,2097100.0,ZTS
2024-08-08,188.300003,1734100.0,ZTS


In [ ]:
#Checking the dtypes of all the columns
print(stocks_data.columns)
print(stocks_data.dtypes)

Index(['Adj Close', 'Volume', 'Company'], dtype='object')
Adj Close    float64
Volume       float64
Company       object
dtype: object


In [ ]:
#find missing companies that were unable to be downloaded
missing_companies = set(com_list['Symbol']) - set(stocks_data['Company'].unique())

missing_companies

{'BF.B', 'BRK.B'}

# Feature Engineering
In this section, we create new features, transform existing ones, and encode categorical variables.

In [ ]:
# Initializing trends_df as an empty fd with the columns I intend to use
trends_df = pd.DataFrame(columns=['Company', 'Slope'])

# Grouping the data by company and performing linear regression to find the slope/trend
for company in stocks_data['Company'].unique():
    if company not in missing_companies: # Check if the company is in the missing_companies set
      company_data = stocks_data[stocks_data['Company'] == company].copy()
      company_data['Days Elapsed'] = (company_data.index - company_data.index[0]).days
      X = company_data['Days Elapsed'].values.reshape(-1, 1)
      y = company_data['Adj Close'].values
      model = LinearRegression()
      model.fit(X, y)

      #creating the temporary df and storing the company name and the trend ie the slope
      temp_df = pd.DataFrame({'Company': company, 'Slope': model.coef_[0]}, index=[0])

      # Concatenate the temporary fd to trends_df
      trends_df = pd.concat([trends_df, temp_df], ignore_index=True)

<ipython-input-115-b1a2aff98ef3>:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  trends_df = pd.concat([trends_df, temp_df], ignore_index=True)


In [ ]:
#display the trends dataframe to see if it valid and in the format needed
trends_df

,Company,Slope
0,MMM,0.015340
1,AOS,0.013026
2,ABT,0.018723
3,ABBV,0.032491
4,ACN,0.054648
...,...,...
496,XYL,0.023502
497,YUM,0.020653
498,ZBRA,0.066224
499,ZBH,0.016747


In [ ]:
market_cap = pd.DataFrame(columns=['Company', 'Market Cap'])

market_cap['Company'] = com_list['Symbol']

for i in market_cap['Company']:
  if i not in missing_companies: # Check if the company symbol is in the missing_companies set
    ticker = yf.Ticker(i) # Pass the company symbol to yf.Ticker
    market_cap.loc[market_cap['Company'] == i, 'Market Cap'] = ticker.info.get('marketCap') # Set the market cap value in the DataFrame

In [ ]:
#Sorting marketcap df by marketcap column
market_cap = market_cap.sort_values(by='Market Cap', ascending=False)

In [ ]:
market_cap

,Company,Market Cap
39,AAPL,3330458255360
320,MSFT,3031639523328
350,NVDA,2713930432512
20,GOOG,2023411023872
19,GOOGL,2022406094848
...,...,...
347,NCLH,6707486208
25,AAL,6396362752
180,ETSY,6197755392
62,BRK.B,NaN


In [ ]:
#renaming market_cap to company_data and adding the slope for each company
company_data = market_cap.copy()
company_data['Slope'] = trends_df['Slope']


In [ ]:
# Ensuring all Nan values are converted to 0
company_data['Market Cap'] = company_data['Market Cap'].fillna(0)

In [ ]:
company_data

,Company,Market Cap,Slope
39,AAPL,3330458255360,0.030276
320,MSFT,3031639523328,0.059700
350,NVDA,2713930432512,0.040961
20,GOOG,2023411023872,0.022939
19,GOOGL,2022406094848,0.022768
...,...,...,...
347,NCLH,6707486208,0.018957
25,AAL,6396362752,0.002395
180,ETSY,6197755392,0.009401
62,BRK.B,0,0.013238


In [ ]:
#renaming Slope to Slope(2008-2024)
company_data = company_data.rename(columns={'Slope': 'Slope(2008-2024)'})

In [ ]:
#add the industry of each company and sort them by industry
company_data['Industry'] = com_list['GICS Sector']
company_data = company_data.sort_values(by='Industry')

In [ ]:
company_data

,Company,Market Cap,Slope(2008-2024),Industry
338,NWSA,15741845504,0.013130,Communication Services
484,DIS,156367552512,0.028472,Communication Services
434,TMUS,227014557696,0.030604,Communication Services
308,MTCH,8858693632,0.043031,Communication Services
356,OMC,18527959040,0.009753,Communication Services
...,...,...,...,...
379,PPL,22520520704,0.020847,Utilities
183,ES,23210369024,0.022037,Utilities
497,XEL,32284882944,0.020653,Utilities
342,NI,14130308096,0.040192,Utilities


In [ ]:
# Group the DataFrame by 'Industry' and calculate the sum of 'Market Cap' for each industry
industry_market_cap = company_data.groupby('Industry')['Market Cap'].sum().reset_index()

# Create a new DataFrame to store the results with 'Others'
industry_market_cap_with_others = pd.DataFrame(columns=['Industry', 'Company', 'Market Cap', 'Slope(2008-2024)'])


In [ ]:
# Iterate over each industry
for industry in company_data['Industry'].unique():
    # Get the top 5 companies for the current industry
    top_5 = company_data[company_data['Industry'] == industry].nlargest(5, 'Market Cap')

    # Calculate the sum of 'Market Cap' for the remaining companies (Others)
    others_market_cap = company_data[(company_data['Industry'] == industry) & (~company_data['Company'].isin(top_5['Company']))]['Market Cap'].sum()

    #calculate sum of slopes for the remaining companies
    others_slope_sum = company_data[(company_data['Industry'] == industry) & (~company_data['Company'].isin(top_5['Company']))]['Slope(2008-2024)'].sum()
    #calculate number of remaining companies
    num_other_companies = len(company_data[(company_data['Industry'] == industry) & (~company_data['Company'].isin(top_5['Company']))])
    #calculate average slope of remaining companies
    others_slope_average = others_slope_sum / num_other_companies if num_other_companies > 0 else 0

    # Append the top 5 companies to the result DataFrame
    industry_market_cap_with_others = pd.concat([industry_market_cap_with_others, top_5[['Industry', 'Company', 'Market Cap', 'Slope(2008-2024)']]])

    # Append the 'Others' row for the current industry
    industry_market_cap_with_others = pd.concat([industry_market_cap_with_others, pd.DataFrame({'Industry': [industry], 'Company': [f'Others-{industry}'], 'Market Cap': [others_market_cap], 'Slope(2008-2024)': [others_slope_average]})])

<ipython-input-127-ea6858a7332a>:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  industry_market_cap_with_others = pd.concat([industry_market_cap_with_others, top_5[['Industry', 'Company', 'Market Cap', 'Slope(2008-2024)']]])


In [ ]:
# Display the DataFrame
industry_market_cap_with_others


,Industry,Company,Market Cap,Slope(2008-2024)
20,Communication Services,GOOG,2023411023872,0.022939
19,Communication Services,GOOGL,2022406094848,0.022768
314,Communication Services,META,1302983737344,0.247081
336,Communication Services,NFLX,273013325824,0.002461
434,Communication Services,TMUS,227014557696,0.030604
...,...,...,...,...
422,Utilities,SO,95271370752,0.018190
159,Utilities,DUK,86787776512,0.013491
125,Utilities,CEG,58935619584,0.007344
26,Utilities,AEP,51557212160,0.013517


# Data Transformation
Here, we apply transformations like scaling and normalization to prepare the data for modeling.


In [ ]:
# Converting market cap to millions and renaming to Market Cap(Millions)
industry_market_cap_with_others['Market Cap'] = industry_market_cap_with_others['Market Cap'] / 1000000
industry_market_cap_with_others = industry_market_cap_with_others.rename(columns={'Market Cap': 'Market Cap(Millions)'})

In [ ]:
#rounding slope to 2dp
industry_market_cap_with_others['Slope(2008-2024)'] = industry_market_cap_with_others['Slope(2008-2024)'].round(5)

#normalising the slope values
industry_market_cap_with_others['Slope(2008-2024)'] = industry_market_cap_with_others['Slope(2008-2024)'] * 100

In [ ]:
# Convert 'Market Cap(Millions)' column to standard Python floats
industry_market_cap_with_others['Market Cap(Millions)'] = industry_market_cap_with_others['Market Cap(Millions)'].astype(float)

#rounding market cap to 2dp
industry_market_cap_with_others['Market Cap(Millions)'] = industry_market_cap_with_others['Market Cap(Millions)'].round(2)

In [ ]:
industry_market_cap_with_others['Entity'] = None

# Map the 'Entity' values to industry_market_cap_with_others
industry_market_cap_with_others['Entity'] = industry_market_cap_with_others['Company'].map(com_list.set_index('Symbol')['Security'])

In [ ]:
industry_market_cap_with_others

,Industry,Company,Market Cap(Millions),Slope(2008-2024),Entity
20,Communication Services,GOOG,2023411.02,2.294,Alphabet Inc. (Class C)
19,Communication Services,GOOGL,2022406.09,2.277,Alphabet Inc. (Class A)
314,Communication Services,META,1302983.74,24.708,Meta Platforms
336,Communication Services,NFLX,273013.33,0.246,Netflix
434,Communication Services,TMUS,227014.56,3.060,T-Mobile US
...,...,...,...,...,...
422,Utilities,SO,95271.37,1.819,Southern Company
159,Utilities,DUK,86787.78,1.349,Duke Energy
125,Utilities,CEG,58935.62,0.734,Constellation Energy
26,Utilities,AEP,51557.21,1.352,American Electric Power


In [ ]:
#finding the average slope of each industry
industry_slope_average = industry_market_cap_with_others.groupby('Industry')['Slope(2008-2024)'].mean().reset_index()

In [ ]:
industry_slope_average

,Industry,Slope(2008-2024)
0,Communication Services,5.784500
1,Consumer Discretionary,1.822833
2,Consumer Staples,2.471833
3,Energy,3.326500
4,Financials,2.367000
5,Health Care,2.151167
6,Industrials,7.456333
7,Information Technology,3.336833
8,Materials,3.737167
9,Real Estate,2.291833


# Conclusion
This section summarizes the data preparation process and provides any final remarks.


In [ ]:
# converting stocks data to csv
stocks_data.to_csv('stocks_data.csv', index=False)
#converting company data to csv
company_data.to_csv('company_data.csv', index=False)
#industry market cap with others to csv
industry_market_cap_with_others.to_csv('industry_market_cap_with_others.csv', index=False)

In [ ]:
#datatypes of the files
print(stocks_data.dtypes)
print(company_data.dtypes)
print(industry_market_cap_with_others.dtypes)

Adj Close    float64
Volume       float64
Company       object
dtype: object
Company              object
Market Cap            int64
Slope(2008-2024)    float64
Industry             object
dtype: object
Industry                 object
Company                  object
Market Cap(Millions)    float64
Slope(2008-2024)        float64
Entity                   object
dtype: object
